# 01. Profiling Dataset ID_REG_KG_2511

Notebook ini bertujuan untuk melakukan profiling awal pada dataset Parquet `ID_REG_KG_2511` menggunakan **Polars** dan **DuckDB** untuk memenuhi target **Milestone 1**.

In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import glob
import polars as pl
import duckdb

print("Versi Polars:", pl.__version__)
print("Versi DuckDB:", duckdb.__version__)

Versi Polars: 1.44.2
Versi DuckDB: 1.5.6


In [3]:
DATA_PATH = "data/raw/ID_REG_KG_2511/*.parquet"

if not glob.glob(DATA_PATH):
    DATA_PATH = "../data/raw/ID_REG_KG_2511/*.parquet"

files = sorted(glob.glob(DATA_PATH))
print(f"Path terdeteksi          : {DATA_PATH}")
print(f"Jumlah berkas Parquet    : {len(files)} berkas")

Path terdeteksi          : ../data/raw/ID_REG_KG_2511/*.parquet
Jumlah berkas Parquet    : 75 berkas


In [4]:
# 1. Ambil skema dari berkas pertama saja
first_file = files[0]
sample_schema = pl.read_parquet(first_file, n_rows=10)

print("--- Skema Data (Polars) ---")
print(sample_schema.schema)

# 2. Hitung total baris secara bertahap (satu per satu file) agar RAM tidak naik
total_rows = 0
for f in files:
    total_rows += pl.scan_parquet(f).select(pl.len()).collect().item()

print(f"\nTotal Baris Keseluruhan Dataset: {total_rows:,}")

--- Skema Data (Polars) ---
Schema({'global_id': Int64, 'local_id': Int64, 'regulation_type': String, 'enacting_body': String, 'regulation_number': String, 'year': String, 'about': String, 'effective_date': String, 'chapter': String, 'article': String, 'content': String, 'chunk_id': Int64, 'embedding': List(Float64), 'tfidf_vector': List(Float64), '__index_level_0__': Int64, 'kg_entities_json': String, 'kg_entity_count': Int64, 'kg_cross_references_json': String, 'kg_cross_ref_count': Int64, 'kg_legal_domains_json': String, 'kg_primary_domain': String, 'kg_domain_confidence': Float64, 'kg_concept_clusters_json': String, 'kg_cluster_count': Int64, 'kg_cluster_diversity': Float64, 'kg_authority_score': Float64, 'kg_hierarchy_level': Int64, 'kg_temporal_score': Float64, 'kg_years_old': Int64, 'kg_legal_richness': Float64, 'kg_legal_complexity': Float64, 'kg_legal_actions_json': String, 'kg_action_types_json': String, 'kg_has_obligations': Boolean, 'kg_has_prohibitions': Boolean, 'kg_has_p

In [5]:
# Membaca 10.000 baris dari berkas pertama untuk mengecek nilai null
sample_10k = pl.read_parquet(first_file, n_rows=10_000)

print("--- Jumlah Nilai Null per Kolom (Sampel 10k Baris) ---")
print(sample_10k.null_count())

--- Jumlah Nilai Null per Kolom (Sampel 10k Baris) ---
shape: (1, 41)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ global_id ┆ local_id ┆ regulatio ┆ enacting_ ┆ … ┆ kg_sancti ┆ kg_concep ┆ kg_comple ┆ kg_connec │
│ ---       ┆ ---      ┆ n_type    ┆ body      ┆   ┆ on_types_ ┆ t_vector_ ┆ teness_sc ┆ tivity_sc │
│ u32       ┆ u32      ┆ ---       ┆ ---       ┆   ┆ json      ┆ json      ┆ ore       ┆ ore       │
│           ┆          ┆ u32       ┆ u32       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆          ┆           ┆           ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0         ┆ 0        ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
└───────────┴──────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘


In [7]:
con = duckdb.connect()

# Batasi penggunaan RAM DuckDB maksimal 1 GB
con.execute("SET max_memory = '1GB'")

print(f"Menjalankan SUMMARIZE pada berkas: {os.path.basename(first_file)}")

# Menggunakan .pl() agar dikonversi langsung ke Polars DataFrame
summary_df = con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{first_file}')").pl()

summary_df

Menjalankan SUMMARIZE pada berkas: train-00001-of-99999.parquet


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""global_id""","""BIGINT""","""0""","""9999""",8571,"""4999.5""","""2886.8956799071675""","""2500""","""5000""","""7500""",10000,0.00
"""local_id""","""BIGINT""","""0""","""9999""",8571,"""4999.5""","""2886.8956799071675""","""2500""","""5000""","""7500""",10000,0.00
"""regulation_type""","""VARCHAR""","""PERATURAN BUPATI""","""Unknown""",12,null,null,null,null,null,10000,0.00
"""enacting_body""","""VARCHAR""","""AGAM""","""WALIKOTA TANGERANG SELATAN PER…",81,null,null,null,null,null,10000,0.00
"""regulation_number""","""VARCHAR""","""-""","""URUT""",134,null,null,null,null,null,10000,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""kg_sanctions_json""","""VARCHAR""","""{""administrative"": [""Denda adm…","""{}""",619,null,null,null,null,null,10000,0.00
"""kg_sanction_types_json""","""VARCHAR""","""[""administrative"", ""civil"", ""m…","""[]""",56,null,null,null,null,null,10000,0.00
"""kg_concept_vector_json""","""VARCHAR""","""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0,…","""[1.0, 0.0, 0.0, 0.0, 0.0, 0.0,…",4041,null,null,null,null,null,10000,0.00
